# Ground state of the 2D Fermi-Hubbard model benchmark

This notebook benchmarks quantum phase estimation (QPE) for the ground state of the
**Fermi-Hubbard model on a two-dimensional square lattice**, and estimates the fault-tolerant
resources required to run it across a sweep of lattice sizes.

$$
H = -T\sum_{\langle i,j\rangle,\;\sigma\in\{\uparrow,\downarrow\}} \left(c^{\dagger}_{i\sigma}c_{j\sigma} + \text{h.c.}\right)
  \; + \; U\sum_{i} n_{i\uparrow}\,n_{i\downarrow},
\qquad n_{i\sigma}=c^{\dagger}_{i\sigma}c_{i\sigma}
$$

Here $c^{\dagger}_{i\sigma}$ and $c_{j\sigma}$ are the fermionic creation and annihilation operators on lattice sites $i,j$, and $n_{i\sigma}$ is the on-site occupation with spin $\sigma$.

**Benchmark specification**

| Quantity | Value |
|---|---|
| Lattice | 2D square, periodic in both directions, $N=L^{2}$ sites |
| Parameters | $U/T = 8$; filling $n = 0.875$ electrons per site $ |
| Target | Ground-state energy to $0.0051\,T$ per site |
| Hardware model | Majorana qubits, measurement error rate $10^{-5}$ |

**References**

- Kivlichan, Ian D., et al. "Improved fault-tolerant quantum simulation of condensed-phase correlated electrons via Trotterization." *Quantum* **4** (2020): 296. [arXiv:1902.10673](https://arxiv.org/abs/1902.10673)
- Campbell, Earl T. "Early fault-tolerant simulations of the Hubbard model." *Quantum Science & Technology* **7**.1 (2022): 015007. [arXiv:2012.09238v4](https://arxiv.org/abs/2012.09238v4)
- Bärtschi, Andreas, et al. "Potential applications of quantum computing at Los Alamos National Laboratory." (2024), Chapter 4 ("High-temperature superconductivity and exotic properties of Fermi-Hubbard models"), Application 1 ("Zero-Temperature Superconductivity"). [arXiv:2406.06625](https://arxiv.org/abs/2406.06625)

**Requirements**

```bash
pip install 'qdk-chemistry[jupyter,qre]'
```

In [ ]:
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    Circuit,
    LatticeGraph,
    MajoranaMapping,
)
from qdk_chemistry.data.circuit import QsharpFactoryData
from qdk_chemistry.utils import Logger
from qdk_chemistry.utils.model_hamiltonians import create_hubbard_hamiltonian
from qdk_chemistry.utils.qsharp import (
    QSHARP_UTILS,
    create_qsharp_context,
    use_qsharp_context,
)

Logger.set_global_level(Logger.LogLevel.off)

HOPPING_T = 1.0                                          # T > 0, energies are quoted in units of T
U_OVER_T = 8.0                                          # U = 8t, strong coupling: LANL Fermi-Hubbard chapter (arXiv:2406.06625 Ch. 4); also Campbell Table II (arXiv:2012.09238v4)
COULOMB_U = U_OVER_T * HOPPING_T
FILLING = 0.875                                          # electrons per SITE (Kivlichan arXiv:1902.10673 Sec. 3.2; half-filling is n=1); round(FILLING * N) electrons total
TARGET_PRECISION_PER_SITE = 0.0051   # per-site ground-state ENERGY target (units of T) chosen for this benchmark; not from LANL App. 1, which specifies accuracy on the order parameter (1e-5)
BENCHMARK_LATTICE_SIZES = [
    size for size in list(range(4, 11, 2)) + list(range(20, 201, 10))
]
EXAMPLE_NUM_SITES = 200
QSHARP_CONTEXT = create_qsharp_context()

def target_precision(size: int) -> float:
    """Absolute ground-state energy accuracy required of an L x L lattice."""
    return TARGET_PRECISION_PER_SITE * size * size

print(f"Hubbard model: U/T = {U_OVER_T:g}  (T = {HOPPING_T:g}, U = {COULOMB_U:g}) Filling n = {FILLING:g} electrons/site")
print(f"Benchmark lattice sizes L = {BENCHMARK_LATTICE_SIZES}")

## The lattice and the qubit Hamiltonian

`LatticeGraph.square` builds the $L\times L$ lattice and `create_hubbard_hamiltonian` turns it into a
fermionic Hamiltonian. The Jordan-Wigner mapping then produces the qubit Hamiltonian on $2N$ qubits
(one per spin-orbital).

In [ ]:
def qubit_operator(size: int):
    """Jordan-Wigner qubit Hamiltonian of the periodic size x size Hubbard lattice."""
    num_sites = size * size
    lattice = LatticeGraph.square(size, size, periodic_x=True, periodic_y=True)
    hamiltonian = create_hubbard_hamiltonian(
        lattice, epsilon=0.0, t=HOPPING_T, U=COULOMB_U
    )
    operator = create("qubit_mapper").run(
        hamiltonian, mapping=MajoranaMapping.jordan_wigner(2 * num_sites)
    )
    return operator

def num_electrons(size: int) -> int:
    """Electron count nearest to the requested per-site filling: round(n * N)."""
    return round(FILLING * size * size)

example_operator = qubit_operator(EXAMPLE_NUM_SITES)
print(f"{EXAMPLE_NUM_SITES}x{EXAMPLE_NUM_SITES} lattice: {example_operator.num_qubits} qubits, {len(example_operator.pauli_strings)} Pauli terms, "
      f"{num_electrons(EXAMPLE_NUM_SITES)} electrons, lambda = {example_operator.schatten_norm:g}")

## Sizing the QPE and Trotter parameters


In [ ]:
import math

TROTTER_ORDER = 2                    # Suzuki-Trotter product-formula order
QPE_FAILURE_PROBABILITY = 0.1        # 1 - confidence that the readout meets the target precision
WEIGHT_THRESHOLD = 1e-12             # Pauli coefficients below this are dropped

one_norm = example_operator.schatten_norm
energy_budget = target_precision(EXAMPLE_NUM_SITES)

# Two independent error sources, so split the energy budget evenly between them.
qpe_budget = energy_budget / 2
trotter_budget = energy_budget / 2   # plaquette `target_accuracy` is itself an energy tolerance

# H*t_0 spectrum fits in [-pi, pi]; the endpoints +/-lambda alias, but the ground state sits strictly inside.
evolution_time = math.pi / one_norm
# m bits resolve dE = 2*pi/(t_0 * 2**m) = 2*lambda/2**m. The second term is the
# Nielsen & Chuang confidence margin ceil(log2(2 + 1/(2*delta))) for failure probability delta.
resolution_bits = math.ceil(math.log2(2 * one_norm / qpe_budget)) + math.ceil(
    math.log2(2 + 1 / (2 * QPE_FAILURE_PROBABILITY))
)

IQPE_ITERATION = 0                   # iteration 0 carries the largest power, 2**(resolution_bits-1)


def reference_state_prep(num_sites: int, electrons: int) -> Circuit:
    """Occupation-number determinant, one X gate per occupied spin-orbital.

    Electrons are split as evenly as possible between the spin-up block
    (qubits 0..N-1) and the spin-down block (qubits N..2N-1).
    """
    num_up = (electrons + 1) // 2
    num_down = electrons // 2
    occupations = (
        [1] * num_up + [0] * (num_sites - num_up)
        + [1] * num_down + [0] * (num_sites - num_down)
    )
    with use_qsharp_context(QSHARP_CONTEXT):
        state_preparation = QSHARP_UTILS.StatePreparation
        params = state_preparation.SingleReferenceParams(bitStrings=occupations, numQubits=2 * num_sites)
        return Circuit(
            qsharp_factory=QsharpFactoryData(
                program=state_preparation.MakeSingleReferenceStateCircuit, parameter=vars(params)
            ),
            qsharp_op=state_preparation.MakePrepareSingleReferenceStateOp(params),
            encoding="jordan-wigner",
        )


def qpe_circuit(
    operator, evolution_time: float, initial_state: Circuit, lattice_size: int, num_bits: int
) -> Circuit:
    """Single IQPE iteration for `operator`, Trotterized to `trotter_budget`.

    `num_iteration` selects one round instead of the whole ladder, so exactly one
    controlled unitary is compiled rather than `num_bits` of them.
    """
    with use_qsharp_context(QSHARP_CONTEXT):
        builder = create(
            "qpe_circuit_builder",
            "qdk_iterative",
            num_bits=num_bits,
            unitary_builder=AlgorithmRef(
                "hamiltonian_unitary_builder",
                "plaquette",
                order=TROTTER_ORDER,
                time=evolution_time,
                target_accuracy=trotter_budget,
                lattice_width=lattice_size,
                lattice_height=lattice_size,
                weight_threshold=WEIGHT_THRESHOLD,
            ),
            controlled_circuit_mapper=AlgorithmRef("controlled_circuit_mapper", "pauli_sequence"),
            num_iteration=IQPE_ITERATION,
        )
        return builder.run(initial_state, operator)[0]


print(f"lambda = {one_norm:g}; energy budget = {energy_budget:g} "
      f"(QPE {qpe_budget:g} + Trotter {trotter_budget:g})")
print(f"t_0 = {evolution_time:g}; resolution bits m = {resolution_bits} "
      f"(largest power 2**{resolution_bits - 1})")

circuit = qpe_circuit(
    example_operator,
    evolution_time,
    reference_state_prep(EXAMPLE_NUM_SITES * EXAMPLE_NUM_SITES, num_electrons(EXAMPLE_NUM_SITES)),
    EXAMPLE_NUM_SITES,
    resolution_bits,
)


## Physical resource estimation

The Q# circuit goes straight to `qdk.qre`; the tracer reports the rotation and measurement counts as
compressed `repeat` blocks, so no logical-count formula is needed.

The trace query expands fine-grained rotations into T gates (`PSSPC`) and schedules the logical operations
with lattice surgery (`LatticeSurgery`); `slow_down_factor` trades runtime for fewer magic-state factories.
The ISA query supplies the surface-code instruction (`ThreeAux`) and a generic distillation model
(`RoundBasedFactory`).

The sweep ranges are given explicitly rather than left at their defaults: with $\sim 10^{7}$ rotations in the
QPE ladder, each rotation must be synthesized far more accurately than the defaults allow, and the default
query returns no result that meets `max_error`.

In [ ]:
# this cell takes ~3 mins to run
from qdk.qre import LatticeSurgery, PSSPC, estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

MAJORANA_ERROR_RATE = 1e-5
ARCHITECTURE = Majorana(error_rate=MAJORANA_ERROR_RATE)
MAX_ERROR = 0.01                     # max allowed error of the estimate; 0.5 was a very loose 50% tolerance

def estimate_physical(circuit: Circuit, name: str):
    """Qubit/runtime Pareto frontier for the given circuit on the Majorana architecture.

    Tracer memory grows like 2**num_bits * num_divisions * n_terms, which is what
    the timeout guards against.
    """
    application = circuit.get_qre_application()
    trace_query = (
        application.q()
        * PSSPC.q(num_ts_per_rotation=list(range(20, 45, 2)))
        * LatticeSurgery.q(slow_down_factor=[1.0 * j for j in range(1, 20)])
    )
    isa_query = ThreeAux.q() * RoundBasedFactory.q(code_query=ThreeAux.q())
    return estimate(application, ARCHITECTURE, isa_query, trace_query, max_error=MAX_ERROR, name=name)

estimates = estimate_physical(circuit, f"{EXAMPLE_NUM_SITES}x{EXAMPLE_NUM_SITES} lattice")
print(estimates)